# Changing Neutrino Parameters in Cosmology Backends

This notebook validates the neutrino interface in different cloelib cosmology backends.
Each backend is validated against their original library.

We also validate that that different backends return the same results for the same neutrino parameters like $\Omega_\nu$ and $N_{\rm eff}$.

For each of the backends, we check the following:
- $H(z)$
- $\chi(z)$
- $P^{\rm lin}(k)$
- $P^{\rm nl}(k)$

Showing the quantities and their relative percent error:  $\delta = \frac{A_{\rm cloelib} - A_{\rm original}}{A_{\rm original}} \times 100\%$

The models we will validate are:
- `mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`
- `mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrinos)
- `mnu = 0.02, 0.04, 0.06`, `N_ur = 3.044`, `N_mnu = 3` (non-degenerate neutrinos)

### Backends validated in this notebook
- cloelib[camb] ✅
- cloelib[class] ✅
- comet ✅
- HMCode2020Emu ✅
- Jax Cosmology ✅

<!-- <style>
details summary {
  font-size: 2.25em;    /* roughly H3 size */
  font-weight: bold;
  cursor: pointer;
}
</style>
<!-- <details> -->
<!-- <summary><strong>📑 Notebook Table of Contents</strong></summary> --> -->
## 📑 Notebook Table of Contents

- [Set-up](#Set-up)
  - [Fiducial Cosmology](#fiducial-cosmology)
- [CLASS Comparison](#class-comparison)
  - [CLASS Set-up](#class-set-up)
  - [cloelib[class] Set-up](#cloelibclass-set-up)
  - [CLASS Comparison: Plots](#class-comparison-plots)
    - [CLASS Comparison: $H(z)$](#class-comparison-hz)
    - [CLASS Comparison: $\chi(z)$](#class-comparison-chiz)
    - [CLASS Comparison: $P^{lin}(k)$](#class-comparison-plink)
    - [CLASS Comparison: $P^{nl}(k)$](#class-comparison-pnlk)
- [CAMB Comparison](#camb-comparison)
  - [CAMB Set-up](#camb-set-up)
  - [cloelib[camb] Set-up](#cloelibcamb-set-up)
  - [CAMB Comparison: Plots](#camb-comparison-plots)
    - [CAMB Comparison: $H(z)$](#camb-comparison-hz)
    - [CAMB Comparison: $\chi(z)$](#camb-comparison-chiz)
    - [CAMB Comparison: $P^{lin}(k)$](#camb-comparison-plink)
    - [CAMB Comparison: $P^{nl}(k)$](#camb-comparison-pnlk)
- [Cross Comparison](#cross-comparison)
  - [Comparison: $H(z)$](#comparison-hz)
  - [Comparison: $\chi(z)$](#comparison-chiz)
  - [Comparison: $P^{lin}(k)$](#comparison-plink)
  - [Comparison: $P^{nl}(k)$](#comparison-pnlk)
- [Comet Tests](#comet-tests)
  - [Comet EFT Set-up](#comet-eft-set-up)
  - [Comet VDG Set-up](#comet-vdg-set-up)
- [HMCode2020Emu Tests](#hmcode2020emu-tests)
  - [HMCode2020Emu Set-up](#hmcode2020emu-set-up)
  - [HMCode2020Emu Comparison: Plots](#hmcode2020emu-comparison-plots)
- [Jax Cosmology Tests](#jax-cosmology-tests)
  - [Jax Cosmology Set-up](#jaxcosmology-set-up)
<!-- </details> -->

# Set-up
This section sets up the notebook, imports the necessary libraries, and defines the fiducial cosmology parameters.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

# cloelib imports
from cloelib.cosmology.camb_cosmology import (
    CAMBBackground,
    CAMBLinearPerturbations,
    CAMBNonLinearPerturbations
)
from cloelib.cosmology.class_cosmology import (
    CLASSBackground,
    CLASSLinearPerturbations,
    CLASSNonLinearPerturbations
)

from cloelib.cosmology.jax_cosmology import (
    JAXBackground,
    JAXLinearPerturbations,
    JAXNonLinearPerturbations
)


from cloelib.auxiliary.units import SPEED_OF_LIGHT

c0 = SPEED_OF_LIGHT/1000

# Plot style
sns.set_theme(style="ticks")
sns.set_palette(sns.color_palette("Paired"))

plt.rc('mathtext', fontset='stix')
plt.rc('xtick',labelsize=20)
plt.rc('ytick',labelsize=20)
plt.rc('font',size=20)
plt.rc('axes', titlesize=25)
plt.rc('axes', labelsize=20)
plt.rc('lines', linewidth=3)
plt.rc('lines', markersize=6)
plt.rc('legend', fontsize=14)

## Fiducial Cosmology
- $\Lambda$ CDM
- No massive neutrinos, i.e., $\sum m_\nu = 0$, $N_{\rm eff} = 3.044$, $N_{\rm m\nu} = 0$
- Mead2020 for the non-linear power spectrum

_Same cosmology as in the `validation_ccl.ipynb` notebook._

In [ ]:
cosmo_dict = {
    'H0': 67.7,
    'Omega_c': 0.25,
    'Omega_b': 0.05,
    'Omega_k':0,
    'omch2': 0.25*0.677**2,
    'ombh2': 0.05*0.677**2,
    'w0': -1,
    'wa':0,
    'ns':0.95,
    'As':2e-9,
    'mnu':0.0,
    'nnu':3.044,
    'N_eff':3.044,
    'halofit_version':'mead2020'
}

# Redshift array to compute power spectra and define n(z)'s
zs = np.linspace(0.0, 3., 100) 
# fourier modes
ks = np.logspace(np.log10(1e-4), np.log10(3.0), 100)

# CLASS comparison

In this section we will validate the CLASS backend for the neutrino interface in `cloelib`.

## Class Set-up

### Model 1 
`mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`

In [ ]:
from classy import Class
import classy

print('Using CLASS %s'%(classy.__version__))

# Linear power spectrum parameters for CLASS
class_fiducial_params = {
    'output': 'mPk,mTk',
    'k_per_decade_for_bao': 70,
    'k_per_decade_for_pk': 10,
    'non linear': 'none',
    'z_max_pk': np.max(zs),
    'P_k_max_1/Mpc': np.max(ks),
    # fiducial cosmology parameters
    'H0': cosmo_dict['H0'],
    'omega_cdm': cosmo_dict['omch2'],
    'omega_b': cosmo_dict['ombh2'],
    'n_s': cosmo_dict['ns'],
    'A_s': cosmo_dict['As'],
    'Omega_k': cosmo_dict['Omega_k'],
    'use_ppf': 'yes',
    'Omega_Lambda': 0.0,
    'w0_fld': cosmo_dict['w0'],
    'wa_fld': cosmo_dict['wa'],
    'N_ur': cosmo_dict['N_eff'],
    'N_ncdm': 0,  # No massive neutrinos
}

# Linear Model 1
results_class_M1 = Class()
results_class_M1.set(class_fiducial_params)
results_class_M1.compute()

# Non-linear power spectrum parameters for CLASS
class_fiducial_nonlinear_params = deepcopy(class_fiducial_params)
class_fiducial_nonlinear_params['non linear'] = 'hmcode'
class_fiducial_nonlinear_params['nonlinear_min_k_max'] = 50
class_fiducial_nonlinear_params['hmcode_tol_sigma'] = 1e-8

# Non-linear Model 1
results_class_M1_nl = Class()
results_class_M1_nl.set(class_fiducial_nonlinear_params)
results_class_M1_nl.compute()


# quantities of interest
class_M1_Hubble = np.array([results_class_M1.Hubble(z) for z in zs]) * c0  # km/s/Mpc
class_M1_chi = np.array([results_class_M1.comoving_distance(z) for z in zs])
class_M1_pk = np.array([[results_class_M1.pk(ki, zi) for ki in ks] for zi in zs])
class_M1_pk_nl = np.array([[results_class_M1_nl.pk(ki, zi) for ki in ks] for zi in zs])


### Model 2
`mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrinos)

In [ ]:
%%time
class_M2_params = deepcopy(class_fiducial_params)
class_M2_params['m_ncdm'] = '0.12'
class_M2_params['N_ncdm'] = 1
class_M2_params['N_ur'] = 2.0308

# Linear Model 2
results_class_M2 = Class()
results_class_M2.set(class_M2_params)
results_class_M2.compute()

# Non-linear power spectrum parameters for CLASS M2
class_M2_nonlinear_params = deepcopy(class_M2_params)
class_M2_nonlinear_params['non linear'] = 'hmcode'
class_M2_nonlinear_params['nonlinear_min_k_max'] = 50
class_M2_nonlinear_params['hmcode_tol_sigma'] = 1e-8

# Non-linear Model 1
results_class_M2_nl = Class()
results_class_M2_nl.set(class_M2_nonlinear_params)
results_class_M2_nl.compute()


# quantities of interest
class_M2_Hubble = np.array([results_class_M2.Hubble(z) for z in zs]) * c0  # km/s/Mpc
class_M2_chi = np.array([results_class_M2.comoving_distance(z) for z in zs])
class_M2_pk = np.array([[results_class_M2.pk(ki, zi) for ki in ks] for zi in zs])
class_M2_pk_nl = np.array([[results_class_M2_nl.pk(ki, zi) for ki in ks] for zi in zs])

### Model 3
`mnu = 0.02, 0.04, 0.06`, `N_ur = 0.0044`, `N_mnu = 3` (non degenerate neutrinos)

In [ ]:
%%time
class_M3_params = deepcopy(class_fiducial_params)
class_M3_params['m_ncdm'] = '0.02, 0.04, 0.06'
class_M3_params['N_ncdm'] = 3
class_M3_params['N_ur'] = 0.0044

# higher precision for neutrino masses:
# comment this out if you do not care for higher precision
# this can take a while to run...
class_M3_params['l_max_ncdm'] = 40
class_M3_params['ncdm_fluid_approximation'] = 3
class_M3_params['tol_ncdm_synchronous'] = 1.e-4

# Linear Model 2
results_class_M3 = Class()
results_class_M3.set(class_M3_params)
results_class_M3.compute()

# Non-linear power spectrum parameters for CLASS M2
class_M3_nonlinear_params = deepcopy(class_M3_params)
class_M3_nonlinear_params['non linear'] = 'hmcode'
class_M3_nonlinear_params['nonlinear_min_k_max'] = 50
class_M3_nonlinear_params['hmcode_tol_sigma'] = 1e-8

# Non-linear Model 1
results_class_M3_nl = Class()
results_class_M3_nl.set(class_M3_nonlinear_params)
results_class_M3_nl.compute()


# quantities of interest
class_M3_Hubble = np.array([results_class_M3.Hubble(z) for z in zs]) * c0  # km/s/Mpc
class_M3_chi = np.array([results_class_M3.comoving_distance(z) for z in zs])
class_M3_pk = np.array([[results_class_M3.pk(ki, zi) for ki in ks] for zi in zs])
class_M3_pk_nl = np.array([[results_class_M3_nl.pk(ki, zi) for ki in ks] for zi in zs])

## cloelib[class] set-up

### Model 1 
`mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`

In [ ]:
cloeclass_M1 = CLASSBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=cosmo_dict['mnu'], 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=0,  # No massive neutrinos
    gamma_MG=0.0
)

# background quantities
cloeclass_M1_Hubble = cloeclass_M1.hubble_parameter(zs)
cloeclass_M1_chi = cloeclass_M1.comoving_distance(zs)

cloeclass_M1_linear = CLASSLinearPerturbations(background=cloeclass_M1, redshifts=zs)
cloeclass_M1_nonlinear = CLASSNonLinearPerturbations(background=cloeclass_M1, 
                                                  redshifts=zs, 
                                                  nonlinear_model='hmcode')

cloeclass_M1_pk = cloeclass_M1_linear.matter_power_spectrum(zs, ks)
cloeclass_M1_pk_nl = cloeclass_M1_nonlinear.matter_power_spectrum(zs, ks)

### Model 2
`mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrino)

In [ ]:
cloeclass_M2 = CLASSBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=0.12, 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=1,  # single neutrinos
    gamma_MG=0.0
)

# background quantities
cloeclass_M2_Hubble = cloeclass_M2.hubble_parameter(zs)
cloeclass_M2_chi = cloeclass_M2.comoving_distance(zs)

cloeclass_M2_linear = CLASSLinearPerturbations(background=cloeclass_M2, redshifts=zs)
cloeclass_M2_nonlinear = CLASSNonLinearPerturbations(background=cloeclass_M2, 
                                                     redshifts=zs, 
                                                     nonlinear_model='hmcode')

cloeclass_M2_pk = cloeclass_M2_linear.matter_power_spectrum(zs, ks)
cloeclass_M2_pk_nl = cloeclass_M2_nonlinear.matter_power_spectrum(zs, ks)

### Model 3
`mnu = 0.02, 0.04, 0.06`, `N_ur = 0.0044`, `N_mnu = 3` (non degenerate neutrinos)

In [ ]:
%%time
cloeclass_M3 = CLASSBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=[0.02,0.04,0.06], 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=3, 
    gamma_MG=0.0
)

# Adding higher accuracy for neutrino masses:
# this may take a while so uncomment if you do not want higher accuracy for neutrino masses
cloeclass_M3.interface_args["CLASSparams"]["l_max_ncdm"] = 40
cloeclass_M3.interface_args["CLASSparams"]["ncdm_fluid_approximation"] = 3
cloeclass_M3.interface_args["CLASSparams"]["tol_ncdm_synchronous"] = 1.e-4

# background quantities
cloeclass_M3_Hubble = cloeclass_M3.hubble_parameter(zs)
cloeclass_M3_chi = cloeclass_M3.comoving_distance(zs)

cloeclass_M3_linear = CLASSLinearPerturbations(background=cloeclass_M3, redshifts=zs)
cloeclass_M3_nonlinear = CLASSNonLinearPerturbations(background=cloeclass_M3, 
                                                     redshifts=zs, 
                                                     nonlinear_model='hmcode')

cloeclass_M3_pk = cloeclass_M3_linear.matter_power_spectrum(zs, ks)
cloeclass_M3_pk_nl = cloeclass_M3_nonlinear.matter_power_spectrum(zs, ks)

## CLASS Comparison: Plots

This section plots the results of the CLASS validation for the neutrino interface in `cloelib[class]`.


### CLASS Comparison: $H(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the class M2 and M3 results here for clarity
axs[0].set_title(r'Hubble Parameter', fontsize=25)
axs[0].set_ylabel(r'$H_{\rm}(z)\: [\rm{km/s/Mpc}]$', fontsize=25)
axs[0].plot(zs,  class_M1_Hubble, color='black',  label=r'class (M1)')
axs[0].plot(zs,  cloeclass_M1_Hubble, color='tab:red', ls='--', label=r'cloelib[class] (M1)')
axs[0].plot(zs,  cloeclass_M2_Hubble, color='tab:blue', ls='-.', label=r'cloelib[class] (M2)')
axs[0].plot(zs,  cloeclass_M3_Hubble, color='tab:green', ls=':', label=r'cloelib[class] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(zs, (cloeclass_M1_Hubble - class_M1_Hubble)/class_M1_Hubble*100, color='tab:red', ls='-', label=r'$\rm{cloelib-class}\,$ (M1)')
axs[1].plot(zs, (cloeclass_M2_Hubble - class_M2_Hubble)/class_M2_Hubble*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-class}\,$ (M2)')
axs[1].plot(zs, (cloeclass_M3_Hubble - class_M3_Hubble)/class_M3_Hubble*100, color='tab:green', ls=':', label=r'$\rm{cloelib-class}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)


### CLASS Comparison: $\chi(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the class M2 and M3 results here for clarity
axs[0].set_title(r'Comoving Distance', fontsize=25)
axs[0].set_ylabel(r'$\chi_{\rm}(z)\: [\rm{Mpc}]$', fontsize=25)
axs[0].plot(zs,  class_M1_chi, color='black',  label=r'class (M1)')
axs[0].plot(zs,  cloeclass_M1_chi, color='tab:red', ls='--', label=r'cloelib[class] (M1)')
axs[0].plot(zs,  cloeclass_M2_chi, color='tab:blue', ls='-.', label=r'cloelib[class] (M2)')
axs[0].plot(zs,  cloeclass_M3_chi, color='tab:green', ls=':', label=r'cloelib[class] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(zs, (cloeclass_M1_chi - class_M1_chi)/class_M1_chi*100, color='tab:red', ls='-', label=r'$\rm{cloelib-class}\,$ (M1)')
axs[1].plot(zs, (cloeclass_M2_chi - class_M2_chi)/class_M2_chi*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-class}\,$ (M2)')
axs[1].plot(zs, (cloeclass_M3_chi - class_M3_chi)/class_M3_chi*100, color='tab:green', ls=':', label=r'$\rm{cloelib-class}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### CLASS Comparison: $P^{lin}(k)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  class_M1_pk[0,:], color='black',  label=r'class (M1)')
axs[0].loglog(ks,  cloeclass_M1_pk[0,:], color='tab:red', ls='--', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloeclass_M2_pk[0,:], color='tab:blue', ls='-.', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloeclass_M3_pk[0,:], color='tab:green', ls=':', label=r'cloelib[class] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloeclass_M1_pk[0,:] - class_M1_pk[0,:])/class_M1_pk[0,:]*100, color='tab:red', ls='-', label=r'$\rm{cloelib-class}\,$ (M1)')
axs[1].plot(ks, (cloeclass_M2_pk[0,:] - class_M2_pk[0,:])/class_M2_pk[0,:]*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-class}\,$ (M2)')
axs[1].plot(ks, (cloeclass_M3_pk[0,:] - class_M3_pk[0,:])/class_M3_pk[0,:]*100, color='tab:green', ls=':', label=r'$\rm{cloelib-class}\,$ (M3)')

axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-0.0007, 0.0007])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### CLASS Comparison: $P^{nl}(k)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Non-Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  class_M1_pk_nl[0,:], color='black',  label=r'class (M1)')
axs[0].loglog(ks,  cloeclass_M1_pk_nl[0,:], color='tab:red', ls='--', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloeclass_M2_pk_nl[0,:], color='tab:blue', ls='-.', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloeclass_M3_pk_nl[0,:], color='tab:green', ls=':', label=r'cloelib[class] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloeclass_M1_pk_nl[0,:] - class_M1_pk_nl[0,:])/class_M1_pk_nl[0,:]*100, color='tab:red', ls='-', label=r'$\rm{cloelib-class}\,$ (M1)')
axs[1].plot(ks, (cloeclass_M2_pk_nl[0,:] - class_M2_pk_nl[0,:])/class_M2_pk_nl[0,:]*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-class}\,$ (M2)')
axs[1].plot(ks, (cloeclass_M3_pk_nl[0,:] - class_M3_pk_nl[0,:])/class_M3_pk_nl[0,:]*100, color='tab:green', ls=':', label=r'$\rm{cloelib-class}\,$ (M3)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-0.0005, 0.0005])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

# CAMB Comparison

In this section we will validate the CAMB backend for the neutrino interface in `cloelib`.

## CAMB Set-up

### Model 1
`mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`

In [ ]:
import camb
print('Using CAMB %s'%(camb.__version__))

# Pass cosmology dictionary to CAMB
camb_fiducial_params = camb.set_params(
    H0=cosmo_dict['H0'],
    ombh2=cosmo_dict['ombh2'],
    omch2=cosmo_dict['omch2'],
    mnu=cosmo_dict['mnu'],
    omk=cosmo_dict['Omega_k'],
    As=cosmo_dict['As'],
    ns=cosmo_dict['ns'],
    nnu=cosmo_dict['N_eff'],  # Effective number of neutrino species
    num_massive_neutrinos=0,  # No massive neutrinos
    halofit_version=cosmo_dict['halofit_version']
)
camb_fiducial_params.share_delta_neff = True
camb_fiducial_params.num_massless_neutrinos=cosmo_dict['N_eff'],  # Number of massless neutrinos
camb_fiducial_params.nu_mass_fractions = []
camb_fiducial_params.nu_mass_eigenstates = 0

camb_fiducial_params.set_matter_power(redshifts=zs, kmax=300) 

# Compute linear P(k)
results_camb_M1 = camb.get_results(camb_fiducial_params)

# quantities of interest
camb_M1_Hubble =results_camb_M1.hubble_parameter(zs)
camb_M1_chi = results_camb_M1.comoving_radial_distance(zs)
camb_M1_pk = results_camb_M1.get_matter_power_interpolator(nonlinear=False,hubble_units=False, k_hunit=False, extrap_kmax=300).P(0,ks)
# Compute non-linear P(k) with Mead2020 halofit version
camb_M1_pk_nl = results_camb_M1.get_matter_power_interpolator(nonlinear=True, hubble_units=False, k_hunit=False).P(0,ks)

### Model 2
`mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrinos)

In [ ]:
camb_M2_params = camb.set_params(
    H0=cosmo_dict['H0'],
    ombh2=cosmo_dict['ombh2'],
    omch2=cosmo_dict['omch2'],
    mnu=0.12,
    omk=cosmo_dict['Omega_k'],
    As=cosmo_dict['As'],
    ns=cosmo_dict['ns'],
    num_massive_neutrinos=1,  # one massive neutrino
    halofit_version=cosmo_dict['halofit_version']
)
camb_M2_params.set_dark_energy(w=cosmo_dict['w0'], wa=cosmo_dict['wa'], dark_energy_model='ppf')
camb_M2_params.share_delta_neff = True
camb_M2_params.num_nu_massless=float(cosmo_dict['N_eff'] - 1)  # Number of massless neutrinos
camb_M2_params.nu_mass_fractions = [1.0]
camb_M2_params.nu_mass_degeneracies = [1.0]
camb_M2_params.nu_mass_numbers = [1]

camb_M2_params.set_matter_power(redshifts=zs, kmax=300) 

# Compute linear P(k)
results_camb_M2 = camb.get_results(camb_M2_params)

print("N_eff for M2:", results_camb_M2.Params.N_eff)

# quantities of interest
camb_M2_Hubble =results_camb_M2.hubble_parameter(zs)
camb_M2_chi = results_camb_M2.comoving_radial_distance(zs)
camb_M2_pk = results_camb_M2.get_matter_power_interpolator(nonlinear=False,hubble_units=False, k_hunit=False, extrap_kmax=300).P(0,ks)
# Compute non-linear P(k) with Mead2020 halofit version
camb_M2_pk_nl = results_camb_M2.get_matter_power_interpolator(nonlinear=True, hubble_units=False, k_hunit=False).P(0,ks)

### Model 3
`mnu = 0.02, 0.04, 0.06`, `N_ur = 0.0044`, `N_mnu = 3` (non degenerate neutrinos)

In [ ]:
%%time
camb_M3_params = camb.set_params(
    H0=cosmo_dict['H0'],
    ombh2=cosmo_dict['ombh2'],
    omch2=cosmo_dict['omch2'],
    mnu=0.12,
    omk=cosmo_dict['Omega_k'],
    As=cosmo_dict['As'],
    ns=cosmo_dict['ns'],
    num_massive_neutrinos=3,  # No massive neutrinos
    halofit_version=cosmo_dict['halofit_version']
)
camb_M3_params.set_dark_energy(w=cosmo_dict['w0'], wa=cosmo_dict['wa'], dark_energy_model='ppf')
camb_M3_params.share_delta_neff = True
camb_M3_params.num_nu_massless=float(cosmo_dict['N_eff'] - 3)  # Number of massless neutrinos
camb_M3_params.nu_mass_fractions = [1/6, 1/3, 1/2] 
camb_M3_params.nu_mass_degeneracies = [1.0] * 3
camb_M3_params.nu_mass_numbers = [1] * 3

# extra high accuracy for detailed neutrino mass:
camb_M3_params.Accuracy.AccuracyBoost = 3
camb_M3_params.Accuracy.lAccuracyBoost = 3
camb_M3_params.Accuracy.AccuratePolarization = False
camb_M3_params.Accuracy.k_eta_max_scalar = 10000.0
camb_M3_params.Transfer.kmax = 50.0

camb_M3_params.set_matter_power(redshifts=zs, kmax=300) 

# Compute linear P(k)
results_camb_M3 = camb.get_results(camb_M3_params)

print("N_eff for M3:", results_camb_M3.Params.N_eff)

# quantities of interest
camb_M3_Hubble =results_camb_M3.hubble_parameter(zs)
camb_M3_chi = results_camb_M3.comoving_radial_distance(zs)
camb_M3_pk = results_camb_M3.get_matter_power_interpolator(nonlinear=False,hubble_units=False, k_hunit=False, extrap_kmax=300).P(0,ks)
# Compute non-linear P(k) with Mead2020 halofit version
camb_M3_pk_nl = results_camb_M3.get_matter_power_interpolator(nonlinear=True, hubble_units=False, k_hunit=False).P(0,ks)

## cloelib[camb] Set-up

### Model 1
`mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`

In [ ]:
cloecamb_M1 = CAMBBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=cosmo_dict['mnu'], 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=0,  # No massive neutrinos
    gamma_MG=0.0
)

# background quantities
cloecamb_M1_Hubble = cloecamb_M1.hubble_parameter(zs)
cloecamb_M1_chi = cloecamb_M1.comoving_distance(zs)

cloecamb_M1_linear = CAMBLinearPerturbations(background=cloecamb_M1, redshifts=zs)
cloecamb_M1_nonlinear = CAMBNonLinearPerturbations(background=cloecamb_M1, 
                                                  redshifts=zs, 
                                                  nonlinear_model='mead2020')

cloecamb_M1_pk = cloecamb_M1_linear.matter_power_spectrum(0, ks)
cloecamb_M1_pk_nl = cloecamb_M1_nonlinear.matter_power_spectrum(0, ks)

### Model 2
`mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrino)

In [ ]:
cloecamb_M2 = CAMBBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=0.12, 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=1,  # one massive neutrinos
    gamma_MG=0.0
)

print("N_eff for M2:", cloecamb_M2.N_eff)

# background quantities
cloecamb_M2_Hubble = cloecamb_M2.hubble_parameter(zs)
cloecamb_M2_chi = cloecamb_M2.comoving_distance(zs)

cloecamb_M2_linear = CAMBLinearPerturbations(background=cloecamb_M2, redshifts=zs)
cloecamb_M2_nonlinear = CAMBNonLinearPerturbations(background=cloecamb_M2, 
                                                  redshifts=zs, 
                                                  nonlinear_model='mead2020')

cloecamb_M2_pk = cloecamb_M2_linear.matter_power_spectrum(0, ks)
cloecamb_M2_pk_nl = cloecamb_M2_nonlinear.matter_power_spectrum(0, ks)

### Model 3
`mnu = 0.02, 0.04, 0.06`, `N_ur = 0.0044`, `N_mnu = 3` (non degenerate neutrinos)

In [ ]:
%%time
cloecamb_M3 = CAMBBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=[0.02, 0.04, 0.06],
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=3,  # one massive neutrinos
    gamma_MG=0.0
)

# higher accuracy for more detailed neutrinos:
cloecamb_M3.interface_args["CAMBparams"].Accuracy.AccuracyBoost = 3
cloecamb_M3.interface_args["CAMBparams"].Accuracy.lAccuracyBoost = 3
cloecamb_M3.interface_args["CAMBparams"].Accuracy.k_eta_max_scalar = 10000.0
cloecamb_M3.interface_args["CAMBparams"].Transfer.kmax = 50.0

print("N_eff for M3:", cloecamb_M3.N_eff)

# background quantities
cloecamb_M3_Hubble = cloecamb_M3.hubble_parameter(zs)
cloecamb_M3_chi = cloecamb_M3.comoving_distance(zs)

cloecamb_M3_linear = CAMBLinearPerturbations(background=cloecamb_M3, redshifts=zs)
cloecamb_M3_nonlinear = CAMBNonLinearPerturbations(background=cloecamb_M3, 
                                                  redshifts=zs, 
                                                  nonlinear_model='mead2020')

cloecamb_M3_pk = cloecamb_M3_linear.matter_power_spectrum(0, ks)
cloecamb_M3_pk_nl = cloecamb_M3_nonlinear.matter_power_spectrum(0, ks)

## CAMB Comparison: Plots

### CAMB Comparison: $H(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the class M2 and M3 results here for clarity
axs[0].set_title(r'Hubble Parameter', fontsize=25)
axs[0].set_ylabel(r'$H_{\rm}(z)\: [\rm{km/s/Mpc}]$', fontsize=25)
axs[0].plot(zs,  camb_M1_Hubble, color='black',  label=r'camb (M1)')
axs[0].plot(zs,  cloecamb_M1_Hubble, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].plot(zs,  cloecamb_M2_Hubble, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].plot(zs,  cloecamb_M3_Hubble, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)
axs[1].plot(zs, (cloecamb_M1_Hubble - camb_M1_Hubble)/camb_M1_Hubble*100, color='tab:red', ls='-', label=r'$\rm{cloelib-camb}\,$ (M1)')
axs[1].plot(zs, (cloecamb_M2_Hubble - camb_M2_Hubble)/camb_M2_Hubble*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-class}\,$ (M2)')
axs[1].plot(zs, (cloecamb_M3_Hubble - camb_M3_Hubble)/camb_M3_Hubble*100, color='tab:green', ls=':', label=r'$\rm{cloelib-class}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### CAMB Comparison: $\chi(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the camb M2 and M3 results here for clarity
axs[0].set_title(r'Comoving Distance', fontsize=25)
axs[0].set_ylabel(r'$\chi_{\rm}(z)\: [\rm{Mpc}]$', fontsize=25)
axs[0].plot(zs,  camb_M1_chi, color='black',  label=r'camb (M1)')
axs[0].plot(zs,  cloecamb_M1_chi, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].plot(zs,  cloecamb_M2_chi, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].plot(zs,  cloecamb_M3_chi, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(zs, (cloecamb_M1_chi - camb_M1_chi)/camb_M1_chi*100, color='tab:red', ls='-', label=r'$\rm{cloelib-camb}\,$ (M1)')
axs[1].plot(zs, (cloecamb_M2_chi - camb_M2_chi)/camb_M2_chi*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-camb}\,$ (M2)')
axs[1].plot(zs, (cloecamb_M3_chi - camb_M3_chi)/camb_M3_chi*100, color='tab:green', ls=':', label=r'$\rm{cloelib-camb}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### CAMB Comparison: $P^{lin}(k)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  camb_M1_pk, color='black',  label=r'camb (M1)')
axs[0].loglog(ks,  cloecamb_M1_pk, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].loglog(ks,  cloecamb_M2_pk, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].loglog(ks,  cloecamb_M3_pk, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloecamb_M1_pk - camb_M1_pk)/camb_M1_pk*100, color='tab:red', ls='-', label=r'$\rm{cloelib-camb}\,$ (M1)')
axs[1].plot(ks, (cloecamb_M2_pk - camb_M2_pk)/camb_M2_pk*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-camb}\,$ (M2)')
axs[1].plot(ks, (cloecamb_M3_pk - camb_M3_pk)/camb_M3_pk*100, color='tab:green', ls=':', label=r'$\rm{cloelib-camb}\,$ (M3)')

axs[1].legend()
axs[1].set_xscale('log')
#axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### CAMB Comparison: $P^{nl}(k)$

In [ ]:

fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Non-Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  camb_M1_pk_nl, color='black',  label=r'camb (M1)')
axs[0].loglog(ks,  cloecamb_M1_pk_nl, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].loglog(ks,  cloecamb_M2_pk_nl, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].loglog(ks,  cloecamb_M3_pk_nl, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloecamb_M1_pk_nl - camb_M1_pk_nl)/camb_M1_pk_nl*100, color='tab:red', ls='-', label=r'$\rm{cloelib-camb}\,$ (M1)')
axs[1].plot(ks, (cloecamb_M2_pk_nl - camb_M2_pk_nl)/camb_M2_pk_nl*100, color='tab:blue', ls='--', label=r'$\rm{cloelib-camb}\,$ (M2)')
axs[1].plot(ks, (cloecamb_M3_pk_nl - camb_M3_pk_nl)/camb_M3_pk_nl*100, color='tab:green', ls=':', label=r'$\rm{cloelib-camb}\,$ (M3)')


axs[1].legend()
#axs[1].set_xscale('log')
axs[1].set_ylim([-0.15, 0.15])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

# Cross Comparison

In the following plots we are taking `cloelib[class]` as the basis and showing the relative percent difference between the other backends and `cloelib[class]`. This shows the internal consistency of the `cloelib` library for its different backends. Note that differences are expected due to the different numerical implementations of the backends and different default numerical precisions.

### Comparison: $H(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the class M2 and M3 results here for clarity
axs[0].set_title(r'Hubble Parameter', fontsize=25)
axs[0].set_ylabel(r'$H_{\rm}(z)\: [\rm{km/s/Mpc}]$', fontsize=25)
axs[0].plot(zs,  cloeclass_M1_Hubble, color='tab:red', ls='-', label=r'cloelib[class] (M1)')
axs[0].plot(zs,  cloeclass_M2_Hubble, color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].plot(zs,  cloeclass_M3_Hubble, color='tab:green', ls='-', label=r'cloelib[class] (M3)')
axs[0].plot(zs,  cloecamb_M1_Hubble, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].plot(zs,  cloecamb_M2_Hubble, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].plot(zs,  cloecamb_M3_Hubble, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)
axs[1].plot(zs, (cloecamb_M1_Hubble - cloeclass_M1_Hubble)/cloeclass_M1_Hubble*100,
            color='tab:red', ls='-', label=r'$\rm{cloelib: camb-class}\,$ (M1)')
axs[1].plot(zs, (cloecamb_M2_Hubble - cloeclass_M2_Hubble)/cloeclass_M2_Hubble*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: camb-class}\,$ (M2)')
axs[1].plot(zs, (cloecamb_M3_Hubble - cloeclass_M3_Hubble)/cloeclass_M3_Hubble*100,
            color='tab:green', ls=':', label=r'$\rm{cloelib: camb-class}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

## Comparison: $\chi(z)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

# not showing the camb M2 and M3 results here for clarity
axs[0].set_title(r'Comoving Distance', fontsize=25)
axs[0].set_ylabel(r'$\chi_{\rm}(z)\: [\rm{Mpc}]$', fontsize=25)
axs[0].plot(zs,  cloeclass_M1_chi, color='tab:red', ls='-', label=r'cloelib[class] (M1)')
axs[0].plot(zs,  cloeclass_M2_chi, color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].plot(zs,  cloeclass_M3_chi, color='tab:green', ls='-', label=r'cloelib[class] (M3)')
axs[0].plot(zs,  cloecamb_M1_chi, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].plot(zs,  cloecamb_M2_chi, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].plot(zs,  cloecamb_M3_chi, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$z$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(zs, (cloecamb_M1_chi - cloeclass_M1_chi)/cloeclass_M1_chi*100,
            color='tab:red', ls='-', label=r'$\rm{cloelib: camb-class}\,$ (M1)')
axs[1].plot(zs, (cloecamb_M2_chi - cloeclass_M2_chi)/cloeclass_M2_chi*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: camb-class}\,$ (M2)')
axs[1].plot(zs, (cloecamb_M3_chi - cloeclass_M3_chi)/cloeclass_M3_chi*100,
            color='tab:green', ls=':', label=r'$\rm{cloelib: camb-class}\,$ (M3)')

axs[1].legend()
# axs[1].set_xscale('log')
axs[1].set_ylim([-0.002, 0.002])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

## Comparison: $P^{lin}(k)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M1_pk[0,:], color='tab:red', ls='-', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloeclass_M2_pk[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloeclass_M3_pk[0,:], color='tab:green', ls='-', label=r'cloelib[class] (M3)')
axs[0].loglog(ks,  cloecamb_M1_pk, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].loglog(ks,  cloecamb_M2_pk, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].loglog(ks,  cloecamb_M3_pk, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloecamb_M1_pk - cloeclass_M1_pk[0,:])/cloeclass_M1_pk[0,:]*100,
            color='tab:red', ls='-', label=r'$\rm{cloelib: camb-class}\,$ (M1)')
axs[1].plot(ks, (cloecamb_M2_pk - cloeclass_M2_pk[0,:])/cloeclass_M1_pk[0,:]*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: camb-class}\,$ (M2)')
axs[1].plot(ks, (cloecamb_M3_pk - cloeclass_M3_pk[0,:])/cloeclass_M1_pk[0,:]*100,
            color='tab:green', ls=':', label=r'$\rm{cloelib: camb-class}\,$ (M3)')

axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-0.25, 0.25])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

## Comparison: $P^{nl}(k)$

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Non-Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M1_pk_nl[0, :], color='tab:red', ls='-', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloeclass_M2_pk_nl[0, :], color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloeclass_M3_pk_nl[0, :], color='tab:green', ls='-', label=r'cloelib[class] (M3)')

axs[0].loglog(ks,  cloecamb_M1_pk_nl, color='tab:red', ls='--', label=r'cloelib[camb] (M1)')
axs[0].loglog(ks,  cloecamb_M2_pk_nl, color='tab:blue', ls='-.', label=r'cloelib[camb] (M2)')
axs[0].loglog(ks,  cloecamb_M3_pk_nl, color='tab:green', ls=':', label=r'cloelib[camb] (M3)')

axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloecamb_M1_pk_nl - cloeclass_M1_pk_nl[0, :])/cloeclass_M1_pk_nl[0, :]*100,
            color='tab:red', ls='-', label=r'$\rm{cloelib-camb}\,$ (M1)')
axs[1].plot(ks, (cloecamb_M2_pk_nl - cloeclass_M2_pk_nl[0, :])/cloeclass_M1_pk_nl[0, :]*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib-camb}\,$ (M2)')
axs[1].plot(ks, (cloecamb_M3_pk_nl - cloeclass_M3_pk_nl[0, :])/cloeclass_M1_pk_nl[0, :]*100,
            color='tab:green', ls=':', label=r'$\rm{cloelib-camb}\,$ (M3)')


axs[1].legend()
#axs[1].set_xscale('log')
axs[1].set_ylim([-0.4, 0.4])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

# Comet Tests

In this section we will validate the Comet EFT backend for the neutrino interface in `cloelib`.

Comet can only take a single neutrino mass, so we will validate the following models:
- M2: `mnu = 0.12`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrinos)

## Comet EFT Set-up
Setup based in `tutorials/observables/spectro.ipynb`.

### Model 4 in Cloelib
`mnu = 0.06`, `N_ur = 2.0308`, `N_mnu = 1` (single neutrinos)

In [ ]:
cloecamb_M4 = CAMBBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c']-1e-3,
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=0.24, 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=1,  # one massive neutrinos
    gamma_MG=0.0
)

print("N_eff for M2:", cloecamb_M4.N_eff)

In [ ]:
from cloelib.observables.CometEFT_spectro import CometEFT_SpectroPower
from cloelib.observables.CometVDG_spectro import CometVDG_SpectroPower
from cloelib.summary_statistics.legendre_multipoles import LegendreMultipoles


In [ ]:
RSD_parameters = {
 'b1': 1.412, 'b2': 0.695, 'bG2': -0.156, 'bGam3': 0.323,
 'c0': 30.948, 'c2': 46.233, 'c4': 10.057, 'cnlo': 0.0
}

zs_c = [1.]

pkmu_comet = CometEFT_SpectroPower(cloecamb_M4, RSD_parameters, redshift=zs_c[0])
noise_syst_parameters = {
 'NP0': 1.056, 'NP20': 0.0, 'NP22': 0.0,
 'fout': 0.0, 'sigmaz': 0.0
}

mps_comet = LegendreMultipoles(pkmu_comet, cloecamb_M2, noise_syst_parameters, nbar=1e-4)

k = np.logspace(-3, np.log10(0.3), 101)
pell_comet = mps_comet.power_multipoles(k=k, ells=[0,2,4])
# COMET without AP
pell_comet_noAP = mps_comet.power_multipoles(k=k, ells=[0,2,4], use_AP=False)

In [ ]:
plt.loglog(k, pell_comet['ell0'], label="with AP distortions")
plt.loglog(k, pell_comet_noAP['ell0'], ls='--', label="no AP distortions")
plt.loglog(k, pell_comet['ell2'])
plt.loglog(k, pell_comet_noAP['ell2'], ls='--')
plt.loglog(k, pell_comet['ell4'])
plt.loglog(k, pell_comet_noAP['ell4'], ls='--')
plt.xlabel(r'$k$')
plt.ylabel(r'$P_\ell(k)$')
plt.legend()

## Comet VDG Set-up

In [ ]:
RSD_parameters['avir'] = 3.0
pkmu_comet_vdg = CometVDG_SpectroPower(cloecamb_M4, RSD_parameters, redshift=zs_c) 

mps_comet_vdg = LegendreMultipoles(pkmu_comet_vdg, cloeclass_M2, noise_syst_parameters, nbar=1e-4)

pell_comet_vdg = mps_comet_vdg.power_multipoles(k=k, ells=[0,2,4])

In [ ]:
plt.loglog(k, pell_comet['ell0'], label="EFT")
plt.loglog(k, pell_comet_vdg['ell0'], '--', label="VDG")
plt.loglog(k, pell_comet['ell2'])
plt.loglog(k, pell_comet_vdg['ell2'], '--')
plt.loglog(k, pell_comet['ell4'])
plt.loglog(k, pell_comet_vdg['ell4'], '--')
plt.xlabel(r'$k$')
plt.ylabel(r'$P_\ell(k)$')
plt.legend()

# HMCode2020Emu Tests

## HMCode2020Emu Set-up
This section sets up the HMCode2020Emu backend for the neutrino interface in `cloelib`.

In [ ]:
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations

In [ ]:
cloehm_M2_linear = HMemuLinearPerturbations(cloecamb_M2, zs)
cloehm_M2_nonlinear = HMemuNonLinearPerturbations(background=cloecamb_M2,
                                                  linearperturbations=cloehm_M2_linear, 
                                                  redshifts=zs, )


cloehm_M2_pk = cloehm_M2_linear.matter_power_spectrum(0, ks)
cloehm_M2_pk_nl = cloehm_M2_nonlinear.matter_power_spectrum(0, ks)

## HMCode2020Emu Comparison: Plots

### Linear Power Spectrum 
vs CAMB and CLASS from cloelib

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M2_pk[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloehm_M2_pk[0,:], color='tab:orange', ls='-', label=r'cloelib[HMCode2020Emu] (M2)')
axs[0].loglog(ks,  cloecamb_M2_pk, color='tab:green', ls='-.', label=r'cloelib[camb] (M2)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloehm_M2_pk[0,:] - cloeclass_M2_pk[0,:])/cloeclass_M2_pk[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: hmc-class}\,$ (M1)')
axs[1].plot(ks, (cloehm_M2_pk[0,:] - cloecamb_M2_pk)/cloecamb_M2_pk*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: hmc-camb}\,$ (M2)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-1.0, 1.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

### Non Linear Power Spectrum
vs CAMB and CLASS from cloelib

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Non-Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M2_pk_nl[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloehm_M2_pk_nl[0,:], color='tab:orange', ls='-', label=r'cloelib[HMCode2020Emu] (M2)')
axs[0].loglog(ks,  cloecamb_M2_pk_nl, color='tab:green', ls='-.', label=r'cloelib[camb] (M2)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloehm_M2_pk_nl[0,:] - cloeclass_M2_pk_nl[0,:])/cloeclass_M2_pk_nl[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: hmc-class}\,$ (M1)')
axs[1].plot(ks, (cloehm_M2_pk_nl[0,:] - cloecamb_M2_pk_nl)/cloecamb_M2_pk_nl*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: hmc-camb}\,$ (M2)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-1.0, 1.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

# Jax Cosmology Tests

## Jax Cosmology Set-up
This section sets up the Jax Cosmology backend for the neutrino interface in `cloelib`.

### Model 1 
`mnu = 0.0`, `N_ur = 3.044`, `N_mnu = 0`

In [ ]:
cloejax_M1 = JAXBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=0.0, 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=0,  # no massive neutrinos
    gamma_MG=0.0
)

print("N_eff for M1:", cloejax_M1.N_eff)

# background quantities
cloejax_M1_Hubble = cloejax_M1.hubble_parameter(zs)
cloejax_M1_chi = cloejax_M1.comoving_distance(zs)

cloejax_M1_linear = JAXLinearPerturbations(background=cloejax_M1)
cloejax_M1_nonlinear = JAXNonLinearPerturbations(background=cloejax_M1)

cloejax_M1_pk = cloejax_M1_linear.matter_power_spectrum(zs, ks)
cloejax_M1_pk_nl = cloejax_M1_nonlinear.matter_power_spectrum(zs, ks)

### Model 2 
`mnu = 0.12`, `N_ur = 3.044`, `N_mnu = 1`

In [ ]:
cloejax_M2 = JAXBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=0.12, 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=1,  # one massive neutrinos
    gamma_MG=0.0
)

print("N_eff for M2:", cloejax_M2.N_eff)

# background quantities
cloejax_M2_Hubble = cloejax_M2.hubble_parameter(zs)
cloejax_M2_chi = cloejax_M2.comoving_distance(zs)

cloejax_M2_linear = JAXLinearPerturbations(background=cloejax_M2)
cloejax_M2_nonlinear = JAXNonLinearPerturbations(background=cloejax_M2)

cloejax_M2_pk = cloejax_M2_linear.matter_power_spectrum(zs, ks)
cloejax_M2_pk_nl = cloejax_M2_nonlinear.matter_power_spectrum(zs, ks)

### Model 3
`mnu = 0.02, 0.04, 0.06`, `N_ur = 3.044`, `N_mnu = 3`

In [ ]:
cloejax_M3 = JAXBackground(
    H0=cosmo_dict['H0'], 
    Omega_b0=cosmo_dict['Omega_b'], 
    Omega_cdm0=cosmo_dict['Omega_c'], 
    Omega_k0=cosmo_dict['Omega_k'], 
    As=cosmo_dict['As'], 
    ns=cosmo_dict['ns'], 
    mnu=np.array([0.02, 0.04, 0.06]), 
    w0=cosmo_dict['w0'], 
    wa=cosmo_dict['wa'], 
    N_mnu=3,  # three massive neutrinos
    gamma_MG=0.0
)
cloejax_M3_linear = JAXLinearPerturbations(background=cloejax_M3)
cloejax_M3_nonlinear = JAXNonLinearPerturbations(background=cloejax_M3)

cloejax_M3_pk = cloejax_M3_linear.matter_power_spectrum(zs, ks)
cloejax_M3_pk_nl = cloejax_M3_nonlinear.matter_power_spectrum(zs, ks)

## Jax Cosmology Plots

## Linear power spectrum

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M1_pk[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloejax_M1_pk[0,:], color='tab:orange', ls='-', label=r'cloelib[jax] (M1)')
axs[0].loglog(ks,  cloecamb_M1_pk, color='tab:green', ls='-.', label=r'cloelib[camb] (M1)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloejax_M1_pk[0,:] - cloeclass_M1_pk[0,:])/cloeclass_M1_pk[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: jax-class}\,$ (M1)')
axs[1].plot(ks, (cloejax_M1_pk[0,:] - cloecamb_M1_pk)/cloecamb_M1_pk*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: jax-camb}\,$ (M1)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-15.0, 15.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M2_pk[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M2)')
axs[0].loglog(ks,  cloejax_M2_pk[0,:], color='tab:orange', ls='-', label=r'cloelib[jax] (M2)')
axs[0].loglog(ks,  cloecamb_M2_pk, color='tab:green', ls='-.', label=r'cloelib[camb] (M2)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloejax_M2_pk[0,:] - cloeclass_M2_pk[0,:])/cloeclass_M2_pk[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: jax-class}\,$ (M2)')
axs[1].plot(ks, (cloejax_M2_pk[0,:] - cloecamb_M2_pk)/cloecamb_M2_pk*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: jax-camb}\,$ (M2)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-15.0, 15.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M3_pk[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M3)')
axs[0].loglog(ks,  cloejax_M3_pk[0,:], color='tab:orange', ls='-', label=r'cloelib[jax] (M3)')
axs[0].loglog(ks,  cloecamb_M3_pk, color='tab:green', ls='-.', label=r'cloelib[camb] (M3)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloejax_M3_pk[0,:] - cloeclass_M3_pk[0,:])/cloeclass_M3_pk[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: jax-class}\,$ (M3)')
axs[1].plot(ks, (cloejax_M3_pk[0,:] - cloecamb_M3_pk)/cloecamb_M3_pk*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: jax-camb}\,$ (M3)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-15.0, 15.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)

## Non-Linear Power spectrum

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8,10),  gridspec_kw={'height_ratios':[2,1] }, sharex=True)

axs[0].set_title(r'Non-Linear $P(k, z=0)$', fontsize=25)
axs[0].set_ylabel(r'$P_{\rm}(k)\: [\rm{Mpc}^3]$', fontsize=25)
axs[0].loglog(ks,  cloeclass_M1_pk_nl[0,:], color='tab:blue', ls='-', label=r'cloelib[class] (M1)')
axs[0].loglog(ks,  cloejax_M1_pk_nl[0,:], color='tab:orange', ls='-', label=r'cloelib[jax] (M1)')
axs[0].loglog(ks,  cloecamb_M1_pk_nl, color='tab:green', ls='-.', label=r'cloelib[camb] (M1)')


axs[0].legend()

## Ratios
axs[1].set_xlabel(r'$k\,[\rm{Mpc}^{-1}]$', fontsize=25)
axs[1].set_ylabel(r'$\%\:\rm{diff.}$', fontsize=25)   
axs[1].plot(ks, (cloejax_M1_pk_nl[0,:] - cloeclass_M1_pk_nl[0,:])/cloeclass_M1_pk_nl[0,:]*100,
            color='tab:orange', ls='-', label=r'$\rm{cloelib: jax-class}\,$ (M1)')
axs[1].plot(ks, (cloejax_M1_pk_nl[0,:] - cloecamb_M1_pk_nl)/cloecamb_M1_pk_nl*100,
            color='tab:blue', ls='--', label=r'$\rm{cloelib: jax-camb}\,$ (M1)')


axs[1].legend()
axs[1].set_xscale('log')
axs[1].set_ylim([-15.0, 15.0])
# axs[1].axhline(0, 0, 300, lw=.8, ls='--', color='black')

for i in range(2):
    axs[i].tick_params(direction='in', which='major', length=8, width=1.5, top=True, right=True)
    axs[i].tick_params(direction='in', which='minor', length=4, width=1.5, top=True, right=True)
    axs[i].grid(alpha=0.5)